In [95]:
import os
import copy
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

In [96]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [97]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

Device: cuda


In [98]:
drive_root = "/content/drive/MyDrive"

pos_matches = []
for root, dirs, files in os.walk(drive_root):
    needed = {
        "scenario23_pos_beam_train.csv",
        "scenario23_pos_beam_val.csv",
        "scenario23_pos_beam_test.csv"
    }
    if needed.issubset(set(files)):
        pos_matches.append(root)

print("Position CSV folder candidates:")
for p in pos_matches:
    print(p)

Position CSV folder candidates:
/content/drive/MyDrive/Pos beam


In [99]:
POS_ROOT = "/content/drive/MyDrive/Pos beam"

pos_train_csv = os.path.join(POS_ROOT, "scenario23_pos_beam_train.csv")
pos_val_csv   = os.path.join(POS_ROOT, "scenario23_pos_beam_val.csv")
pos_test_csv  = os.path.join(POS_ROOT, "scenario23_pos_beam_test.csv")

print(os.path.exists(pos_train_csv), pos_train_csv)
print(os.path.exists(pos_val_csv), pos_val_csv)
print(os.path.exists(pos_test_csv), pos_test_csv)

True /content/drive/MyDrive/Pos beam/scenario23_pos_beam_train.csv
True /content/drive/MyDrive/Pos beam/scenario23_pos_beam_val.csv
True /content/drive/MyDrive/Pos beam/scenario23_pos_beam_test.csv


In [100]:
train_pos_df = pd.read_csv(pos_train_csv)
val_pos_df   = pd.read_csv(pos_val_csv)
test_pos_df  = pd.read_csv(pos_test_csv)

print("Train:", train_pos_df.shape)
print("Val  :", val_pos_df.shape)
print("Test :", test_pos_df.shape)

print(train_pos_df.head())
print(train_pos_df.columns.tolist())

Train: (6832, 3)
Val  : (3416, 3)
Test : (1139, 3)
   index                                  unit2_pos  unit1_beam
0   3532    [0.8092883966431671, 0.521083920903955]          17
1   2224  [0.4816276084988933, 0.29434536152734486]          14
2   9416    [0.220278556834608, 0.4136596156292844]          17
3   8510  [0.21412273613497904, 0.4547214157104936]          20
4   6877  [0.14500641727379412, 0.4097884695072434]          17
['index', 'unit2_pos', 'unit1_beam']


In [101]:
label_col = train_pos_df.columns[-1]
feature_cols = [c for c in train_pos_df.columns if c != label_col]

print("Feature columns:", feature_cols)
print("Label column:", label_col)

print("Label min/max:", train_pos_df[label_col].min(), train_pos_df[label_col].max())

Feature columns: ['index', 'unit2_pos']
Label column: unit1_beam
Label min/max: 2 30


In [102]:
import ast

def parse_unit2_pos(pos_value):
    if isinstance(pos_value, str):
        return ast.literal_eval(pos_value)
    return pos_value

for df in [train_pos_df, val_pos_df, test_pos_df]:
    parsed = df["unit2_pos"].apply(parse_unit2_pos)
    df["pos_x"] = parsed.apply(lambda v: float(v[0]))
    df["pos_y"] = parsed.apply(lambda v: float(v[1]))

feature_cols = ["pos_x", "pos_y"]
label_col = "unit1_beam"

print(train_pos_df[["index", "unit2_pos", "pos_x", "pos_y", label_col]].head())
print("Feature columns:", feature_cols)
print("Label column:", label_col)
print("Original label min/max:", train_pos_df[label_col].min(), train_pos_df[label_col].max())
print("Unique labels:", sorted(train_pos_df[label_col].unique()))

   index                                  unit2_pos     pos_x     pos_y  \
0   3532    [0.8092883966431671, 0.521083920903955]  0.809288  0.521084   
1   2224  [0.4816276084988933, 0.29434536152734486]  0.481628  0.294345   
2   9416    [0.220278556834608, 0.4136596156292844]  0.220279  0.413660   
3   8510  [0.21412273613497904, 0.4547214157104936]  0.214123  0.454721   
4   6877  [0.14500641727379412, 0.4097884695072434]  0.145006  0.409788   

   unit1_beam  
0          17  
1          14  
2          17  
3          20  
4          17  
Feature columns: ['pos_x', 'pos_y']
Label column: unit1_beam
Original label min/max: 2 30
Unique labels: [np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24), np.int64(25), np.int64(26), np.in

label remapping 

In [103]:
import ast
import numpy as np

def parse_unit2_pos(pos_value):
    if isinstance(pos_value, str):
        return ast.literal_eval(pos_value)
    return pos_value

for df in [train_pos_df, val_pos_df, test_pos_df]:
    parsed = df["unit2_pos"].apply(parse_unit2_pos)
    df["pos_x"] = parsed.apply(lambda v: float(v[0]))
    df["pos_y"] = parsed.apply(lambda v: float(v[1]))

    df["distance"] = np.sqrt(df["pos_x"]**2 + df["pos_y"]**2)
    df["angle"] = np.arctan2(df["pos_y"], df["pos_x"])
    df["sin_angle"] = np.sin(df["angle"])
    df["cos_angle"] = np.cos(df["angle"])

    df["pos_x2"] = df["pos_x"] ** 2
    df["pos_y2"] = df["pos_y"] ** 2
    df["pos_xy"] = df["pos_x"] * df["pos_y"]

feature_cols = [
    "pos_x",
    "pos_y",
    "distance",
    "sin_angle",
    "cos_angle",
    "pos_x2",
    "pos_y2",
    "pos_xy",
]

label_col = "unit1_beam"

print(train_pos_df[["index", "unit2_pos", *feature_cols, label_col]].head())
print("Feature columns:", feature_cols)
print("Number of features:", len(feature_cols))
print("Label column:", label_col)
print("Original label min/max:", train_pos_df[label_col].min(), train_pos_df[label_col].max())

   index                                  unit2_pos     pos_x     pos_y  \
0   3532    [0.8092883966431671, 0.521083920903955]  0.809288  0.521084   
1   2224  [0.4816276084988933, 0.29434536152734486]  0.481628  0.294345   
2   9416    [0.220278556834608, 0.4136596156292844]  0.220279  0.413660   
3   8510  [0.21412273613497904, 0.4547214157104936]  0.214123  0.454721   
4   6877  [0.14500641727379412, 0.4097884695072434]  0.145006  0.409788   

   distance  sin_angle  cos_angle    pos_x2    pos_y2    pos_xy  unit1_beam  
0  0.962536   0.541365   0.840787  0.654948  0.271528  0.421707          17  
1  0.564450   0.521472   0.853268  0.231965  0.086639  0.141765          14  
2  0.468654   0.882654   0.470023  0.048523  0.171114  0.091120          17  
3  0.502613   0.904714   0.426019  0.045849  0.206772  0.097366          20  
4  0.434688   0.942719   0.333588  0.021027  0.167927  0.059422          17  
Feature columns: ['pos_x', 'pos_y', 'distance', 'sin_angle', 'cos_angle', 'pos_x2

new feature 

In [104]:
import ast
import numpy as np

def parse_unit2_pos(pos_value):
    if isinstance(pos_value, str):
        return ast.literal_eval(pos_value)
    return pos_value

eps = 1e-8

for df in [train_pos_df, val_pos_df, test_pos_df]:
    parsed = df["unit2_pos"].apply(parse_unit2_pos)

    df["pos_x"] = parsed.apply(lambda v: float(v[0]))
    df["pos_y"] = parsed.apply(lambda v: float(v[1]))

    df["distance"] = np.sqrt(df["pos_x"]**2 + df["pos_y"]**2)
    df["distance2"] = df["distance"] ** 2
    df["distance3"] = df["distance"] ** 3

    df["angle"] = np.arctan2(df["pos_y"], df["pos_x"])
    df["sin_angle"] = np.sin(df["angle"])
    df["cos_angle"] = np.cos(df["angle"])

    df["pos_x2"] = df["pos_x"] ** 2
    df["pos_y2"] = df["pos_y"] ** 2
    df["pos_x3"] = df["pos_x"] ** 3
    df["pos_y3"] = df["pos_y"] ** 3
    df["pos_xy"] = df["pos_x"] * df["pos_y"]

    df["unit_x"] = df["pos_x"] / (df["distance"] + eps)
    df["unit_y"] = df["pos_y"] / (df["distance"] + eps)

feature_cols = [
    "pos_x",
    "pos_y",
    "distance",
    "distance2",
    "distance3",
    "sin_angle",
    "cos_angle",
    "pos_x2",
    "pos_y2",
    "pos_x3",
    "pos_y3",
    "pos_xy",
    "unit_x",
    "unit_y",
]

label_col = "unit1_beam"

print("Feature columns:", feature_cols)
print("Number of features:", len(feature_cols))
print("Label column:", label_col)

display(train_pos_df[["index", "unit2_pos", *feature_cols, label_col]].head())
print("Label unique:", train_pos_df[label_col].nunique())
print("Label min/max:", train_pos_df[label_col].min(), train_pos_df[label_col].max())

Feature columns: ['pos_x', 'pos_y', 'distance', 'distance2', 'distance3', 'sin_angle', 'cos_angle', 'pos_x2', 'pos_y2', 'pos_x3', 'pos_y3', 'pos_xy', 'unit_x', 'unit_y']
Number of features: 14
Label column: unit1_beam


,index,unit2_pos,pos_x,pos_y,distance,distance2,distance3,sin_angle,cos_angle,pos_x2,pos_y2,pos_x3,pos_y3,pos_xy,unit_x,unit_y,unit1_beam
0,3532,"[0.8092883966431671, 0.521083920903955]",0.809288,0.521084,0.962536,0.926476,0.891767,0.541365,0.840787,0.654948,0.271528,0.530042,0.141489,0.421707,0.840787,0.541365,17
1,2224,"[0.4816276084988933, 0.29434536152734486]",0.481628,0.294345,0.564450,0.318604,0.179836,0.521472,0.853268,0.231965,0.086639,0.111721,0.025502,0.141765,0.853268,0.521472,14
2,9416,"[0.220278556834608, 0.4136596156292844]",0.220279,0.413660,0.468654,0.219637,0.102934,0.882654,0.470023,0.048523,0.171114,0.010688,0.070783,0.091120,0.470023,0.882654,17
3,8510,"[0.21412273613497904, 0.4547214157104936]",0.214123,0.454721,0.502613,0.252620,0.126970,0.904714,0.426019,0.045849,0.206772,0.009817,0.094023,0.097366,0.426019,0.904714,20
4,6877,"[0.14500641727379412, 0.4097884695072434]",0.145006,0.409788,0.434688,0.188953,0.082136,0.942719,0.333588,0.021027,0.167927,0.003049,0.068814,0.059422,0.333588,0.942719,17


Label unique: 29
Label min/max: 2 30


In [105]:
all_labels = sorted(train_pos_df[label_col].unique())

label_to_id = {label: i for i, label in enumerate(all_labels)}
id_to_label = {i: label for label, i in label_to_id.items()}

num_classes_pos = len(all_labels)

print("Number of position classes:", num_classes_pos)
print("Label mapping:", label_to_id)

Number of position classes: 29
Label mapping: {np.int64(2): 0, np.int64(3): 1, np.int64(4): 2, np.int64(5): 3, np.int64(6): 4, np.int64(7): 5, np.int64(8): 6, np.int64(9): 7, np.int64(10): 8, np.int64(11): 9, np.int64(12): 10, np.int64(13): 11, np.int64(14): 12, np.int64(15): 13, np.int64(16): 14, np.int64(17): 15, np.int64(18): 16, np.int64(19): 17, np.int64(20): 18, np.int64(21): 19, np.int64(22): 20, np.int64(23): 21, np.int64(24): 22, np.int64(25): 23, np.int64(26): 24, np.int64(27): 25, np.int64(28): 26, np.int64(29): 27, np.int64(30): 28}


morm- only pos x,y

In [74]:
train_mean = train_pos_df[feature_cols].astype(float).mean()
train_std  = train_pos_df[feature_cols].astype(float).std().replace(0, 1)

print("Train mean:")
display(train_mean)

print("Train std:")
display(train_std)

Train mean:


,0
pos_x,0.343933
pos_y,0.396131
distance,0.540083
distance2,0.311916
distance3,0.193269
sin_angle,0.744473
cos_angle,0.625777
pos_x2,0.141736
pos_y2,0.170180
pos_x3,0.069200


Train std:


,0
pos_x,0.153132
pos_y,0.115162
distance,0.142230
distance2,0.177328
distance3,0.181966
sin_angle,0.161494
cos_angle,0.167600
pos_x2,0.137104
pos_y2,0.098119
pos_x3,0.107203


correct dataset 

In [106]:
class PositionBeamDataset(Dataset):
    def __init__(self, df, feature_cols, label_col, mean, std, label_to_id):
        self.df = df.reset_index(drop=True).copy()
        self.feature_cols = feature_cols
        self.label_col = label_col
        self.mean = mean
        self.std = std
        self.label_to_id = label_to_id

        x = self.df[self.feature_cols].astype(float)
        x = (x - self.mean) / self.std

        y_raw = self.df[self.label_col].values
        y = [self.label_to_id[int(v)] for v in y_raw]

        self.x = torch.tensor(x.values, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

correct dataloader 

In [107]:
batch_size = 64

dataset_pos_train = PositionBeamDataset(train_pos_df, feature_cols, label_col, train_mean, train_std, label_to_id)
dataset_pos_val   = PositionBeamDataset(val_pos_df, feature_cols, label_col, train_mean, train_std, label_to_id)
dataset_pos_test  = PositionBeamDataset(test_pos_df, feature_cols, label_col, train_mean, train_std, label_to_id)

train_loader_pos = DataLoader(dataset_pos_train, batch_size=batch_size, shuffle=True)
val_loader_pos   = DataLoader(dataset_pos_val, batch_size=batch_size, shuffle=False)
test_loader_pos  = DataLoader(dataset_pos_test, batch_size=batch_size, shuffle=False)

x_batch, y_batch = next(iter(train_loader_pos))

print("x_batch shape:", x_batch.shape)
print("y_batch shape:", y_batch.shape)
print("First x:", x_batch[:5])
print("First mapped y:", y_batch[:10])

input_dim = x_batch.shape[1]
num_classes = num_classes_pos

print("input_dim:", input_dim)
print("num_classes:", num_classes)

x_batch shape: torch.Size([64, 14])
y_batch shape: torch.Size([64])
First x: tensor([[ 0.6498,  0.5952,  0.7187,  0.5675,  0.3941, -0.1302,  0.3854,  0.4004,
          0.4662,  0.1678,  0.2887,  0.8351,  0.3854, -0.1302],
        [-0.8635, -1.5201, -1.6452, -1.2306, -0.9045, -0.1376,  0.3929, -0.7069,
         -1.2363, -0.5570, -0.8822, -1.1001,  0.3929, -0.1376],
        [-0.5376, -2.0916, -1.6584, -1.2371, -0.9074, -1.4496,  1.3973, -0.5346,
         -1.4887, -0.4785, -0.9746, -1.1752,  1.3973, -1.4496],
        [-0.4353,  1.6313,  0.7480,  0.5978,  0.4226,  0.9838, -1.1747, -0.4730,
          1.7414, -0.4467,  1.5814,  0.2989, -1.1747,  0.9838],
        [-0.4361, -0.0519, -0.4324, -0.4674, -0.4597,  0.4382, -0.2784, -0.4735,
         -0.1830, -0.4469, -0.2468, -0.3548, -0.2784,  0.4382]])
First mapped y: tensor([15,  2,  0, 24, 13, 18, 13, 28,  9, 15])
input_dim: 14
num_classes: 29


In [108]:
x_batch, y_batch = next(iter(train_loader_pos))

print("x_batch shape:", x_batch.shape)
print("y_batch shape:", y_batch.shape)
print("First x:", x_batch[:5])
print("First y:", y_batch[:10])

input_dim = x_batch.shape[1]
num_classes = 64

print("input_dim:", input_dim)
print("num_classes:", num_classes)

x_batch shape: torch.Size([64, 14])
y_batch shape: torch.Size([64])
First x: tensor([[-0.8435,  0.1222, -0.5418, -0.5500, -0.5166,  0.8759, -0.9663, -0.6974,
         -0.0195, -0.5531, -0.1208, -0.5983, -0.9663,  0.8759],
        [-0.9343,  0.5208, -0.2933, -0.3583, -0.3819,  1.0571, -1.3290, -0.7395,
          0.3858, -0.5699,  0.2175, -0.5555, -1.3290,  1.0571],
        [ 0.0867,  0.1678,  0.0550, -0.0661, -0.1582,  0.0854,  0.1561, -0.1031,
          0.0247, -0.2204, -0.0856,  0.1346,  0.1561,  0.0854],
        [-0.8091,  0.7717, -0.0527, -0.1594, -0.2319,  1.0290, -1.2686, -0.6806,
          0.6629, -0.5461,  0.4686, -0.3720, -1.2686,  1.0290],
        [ 1.2700,  0.5976,  1.2044,  1.0949,  0.9163, -0.5628,  0.7820,  1.0806,
          0.4688,  0.8104,  0.2911,  1.3732,  0.7820, -0.5628]])
First y: tensor([14, 18, 15, 20, 15, 11, 19, 15, 14, 13])
input_dim: 14
num_classes: 64


4 layer mlp 

newly 

In [109]:
class FourLayerMLP(nn.Module):
    def __init__(self, input_dim, num_classes=64, hidden_dims=(512, 256, 128, 64), dropout=0.10):
        super().__init__()
        h1, h2, h3, h4 = hidden_dims

        self.net = nn.Sequential(
            nn.Linear(input_dim, h1),
            nn.BatchNorm1d(h1),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(h1, h2),
            nn.BatchNorm1d(h2),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(h2, h3),
            nn.BatchNorm1d(h3),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(h3, h4),
            nn.BatchNorm1d(h4),
            nn.GELU(),

            nn.Linear(h4, num_classes)
        )

    def forward(self, x):
        return self.net(x)

In [110]:
model_name = "mlp4"

set_seed(42)

model_pos = build_position_model(
    model_name,
    input_dim=input_dim,
    num_classes=num_classes_pos
)

save_path = f"/content/drive/MyDrive/best_position_{model_name}_engineered_v2.pth"

history_pos = train_position_model(
    model=model_pos,
    train_loader=train_loader_pos,
    val_loader=val_loader_pos,
    device=device,
    epochs=100,
    lr=3e-4,
    weight_decay=1e-6,
    milestones=(50, 80),
    save_path=save_path
)

model_pos_eval = build_position_model(
    model_name,
    input_dim=input_dim,
    num_classes=num_classes_pos
)

model_pos_eval.load_state_dict(torch.load(save_path, map_location=device))

test_metrics_pos = evaluate_topk_pos(
    model_pos_eval,
    test_loader_pos,
    device,
    ks=(1, 2, 3, 5)
)

print(f"{model_name} Test metrics:", test_metrics_pos)

Epoch 01 | Train Loss: 2.6288 | Top1: 45.96 | Top2: 69.79 | Top3: 78.45 | Top5: 88.00
Saved best model
Epoch 02 | Train Loss: 1.9386 | Top1: 50.41 | Top2: 72.98 | Top3: 82.76 | Top5: 91.42
Saved best model
Epoch 03 | Train Loss: 1.6399 | Top1: 53.13 | Top2: 75.00 | Top3: 85.42 | Top5: 92.62
Saved best model
Epoch 04 | Train Loss: 1.5035 | Top1: 53.54 | Top2: 74.94 | Top3: 86.33 | Top5: 94.03
Saved best model
Epoch 05 | Train Loss: 1.4330 | Top1: 55.50 | Top2: 78.02 | Top3: 87.85 | Top5: 94.29
Saved best model
Epoch 06 | Train Loss: 1.3938 | Top1: 55.15 | Top2: 77.25 | Top3: 87.94 | Top5: 94.96
Epoch 07 | Train Loss: 1.3572 | Top1: 55.15 | Top2: 78.13 | Top3: 88.44 | Top5: 95.46
Epoch 08 | Train Loss: 1.3420 | Top1: 56.24 | Top2: 77.37 | Top3: 87.79 | Top5: 95.40
Saved best model
Epoch 09 | Train Loss: 1.3172 | Top1: 57.17 | Top2: 78.83 | Top3: 88.64 | Top5: 95.73
Saved best model
Epoch 10 | Train Loss: 1.2999 | Top1: 54.92 | Top2: 77.28 | Top3: 88.88 | Top5: 95.35
Epoch 11 | Train Loss

In [111]:
class FourLayerMLP(nn.Module):
    def __init__(self, input_dim, num_classes=64, hidden_dims=(128, 256, 128, 64), dropout=0.2):
        super().__init__()
        h1, h2, h3, h4 = hidden_dims

        self.net = nn.Sequential(
            nn.Linear(input_dim, h1),
            nn.BatchNorm1d(h1),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(h1, h2),
            nn.BatchNorm1d(h2),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(h2, h3),
            nn.BatchNorm1d(h3),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(h3, h4),
            nn.BatchNorm1d(h4),
            nn.ReLU(),

            nn.Linear(h4, num_classes)
        )

    def forward(self, x):
        return self.net(x)

ann 

In [112]:
class SimpleANN(nn.Module):
    def __init__(self, input_dim, num_classes=64, hidden_dim=128, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, x):
        return self.net(x)

ann + attention

In [115]:
class ANNWithAttention(nn.Module):
    def __init__(self, input_dim, num_classes=64, hidden_dim=128, dropout=0.2):
        super().__init__()

        self.attn = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.Tanh(),
            nn.Linear(hidden_dim, input_dim),
            nn.Softmax(dim=1)
        )

        self.classifier = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.GeLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim, hidden_dim),
            nn.GeLU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, x):
        weights = self.attn(x)
        x_weighted = x * weights
        return self.classifier(x_weighted)

rnn 

In [116]:
class PositionRNN(nn.Module):
    def __init__(self, input_dim, num_classes=64, hidden_dim=64, num_layers=2, dropout=0.1):
        super().__init__()

        self.rnn = nn.RNN(
            input_size=1,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )

        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x_seq = x.unsqueeze(-1)
        out, h = self.rnn(x_seq)
        last = out[:, -1, :]
        return self.fc(last)

lstm 

In [117]:
class PositionLSTM(nn.Module):
    def __init__(self, input_dim, num_classes=64, hidden_dim=64, num_layers=2, dropout=0.1):
        super().__init__()

        self.lstm = nn.LSTM(
            input_size=1,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )

        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        x_seq = x.unsqueeze(-1)
        out, (h, c) = self.lstm(x_seq)
        last = out[:, -1, :]
        return self.fc(last)

top - k evaluation

In [118]:
def evaluate_topk_pos(model, loader, device, ks=(1, 2, 3, 5)):
    model.eval()
    total = 0
    correct = {k: 0 for k in ks}

    with torch.no_grad():
        for x, labels in loader:
            x = x.to(device)
            labels = labels.to(device)

            outputs = model(x)

            max_k = max(ks)
            _, pred = torch.topk(outputs, k=max_k, dim=1)
            pred = pred.t()

            total += labels.size(0)

            for k in ks:
                correct[k] += pred[:k].eq(labels.view(1, -1)).sum().item()

    return {f"top{k}": 100.0 * correct[k] / total for k in ks}

trainer 

In [119]:
def train_position_model(
    model,
    train_loader,
    val_loader,
    device,
    epochs=50,
    lr=1e-3,
    weight_decay=1e-4,
    milestones=(20, 35),
    save_path="/content/drive/MyDrive/best_position_model.pth"
):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.MultiStepLR(optimizer, milestones=list(milestones), gamma=0.1)

    best_top1 = -1
    history = []

    for epoch in range(1, epochs + 1):
        model.train()
        running_loss = 0.0
        total_train = 0

        for x, labels in train_loader:
            x = x.to(device)
            labels = labels.to(device)

            optimizer.zero_grad()
            outputs = model(x)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            bs = labels.size(0)
            running_loss += loss.item() * bs
            total_train += bs

        scheduler.step()

        train_loss = running_loss / total_train
        val_metrics = evaluate_topk_pos(model, val_loader, device, ks=(1, 2, 3, 5))

        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            **val_metrics
        }
        history.append(row)

        print(
            f"Epoch {epoch:02d} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Top1: {val_metrics['top1']:.2f} | "
            f"Top2: {val_metrics['top2']:.2f} | "
            f"Top3: {val_metrics['top3']:.2f} | "
            f"Top5: {val_metrics['top5']:.2f}"
        )

        if val_metrics["top1"] > best_top1:
            best_top1 = val_metrics["top1"]
            torch.save(copy.deepcopy(model.state_dict()), save_path)
            print("Saved best model")

    return pd.DataFrame(history)

model builder 

In [120]:
def build_position_model(model_name, input_dim, num_classes=num_classes_pos):
    if model_name == "mlp4":
        return FourLayerMLP(input_dim=input_dim, num_classes=num_classes).to(device)

    elif model_name == "ann":
        return SimpleANN(input_dim=input_dim, num_classes=num_classes).to(device)

    elif model_name == "ann_attention":
        return ANNWithAttention(input_dim=input_dim, num_classes=num_classes).to(device)

    elif model_name == "rnn":
        return PositionRNN(input_dim=input_dim, num_classes=num_classes).to(device)

    elif model_name == "lstm":
        return PositionLSTM(input_dim=input_dim, num_classes=num_classes).to(device)

    else:
        raise ValueError("Unknown model_name")

run all the models 

In [121]:
position_model_names = [
    "mlp4",
    "ann",
    "ann_attention",
    "rnn",
    "lstm"
]

all_position_histories = {}
pos_results = []

for model_name in position_model_names:
    print("\n" + "="*80)
    print(f"Training Position Model: {model_name}")
    print("="*80)

    set_seed(42)

    model_pos = build_position_model(
        model_name,
        input_dim=input_dim,
        num_classes=num_classes_pos
    )

    save_path = f"/content/drive/MyDrive/best_position_{model_name}.pth"

    history_pos = train_position_model(
        model=model_pos,
        train_loader=train_loader_pos,
        val_loader=val_loader_pos,
        device=device,
        epochs=100,
        lr=1e-3,
        weight_decay=1e-4,
        milestones=(20, 35),
        save_path=save_path
    )

    all_position_histories[model_name] = history_pos

    model_pos_eval = build_position_model(
        model_name,
        input_dim=input_dim,
        num_classes=num_classes_pos
    )

    model_pos_eval.load_state_dict(torch.load(save_path, map_location=device))

    test_metrics_pos = evaluate_topk_pos(model_pos_eval, test_loader_pos, device, ks=(1, 2, 3, 5))

    print(f"{model_name} Test metrics:", test_metrics_pos)

    pos_results.append({
        "Model": model_name,
        "Top-1": test_metrics_pos["top1"],
        "Top-2": test_metrics_pos["top2"],
        "Top-3": test_metrics_pos["top3"],
        "Top-5": test_metrics_pos["top5"],
    })

    del model_pos
    del model_pos_eval
    torch.cuda.empty_cache()

pos_results_df = pd.DataFrame(pos_results)
pos_results_df


Training Position Model: mlp4
Epoch 01 | Train Loss: 2.1800 | Top1: 47.22 | Top2: 70.78 | Top3: 82.06 | Top5: 91.28
Saved best model
Epoch 02 | Train Loss: 1.6049 | Top1: 51.73 | Top2: 74.03 | Top3: 85.74 | Top5: 93.59
Saved best model
Epoch 03 | Train Loss: 1.4920 | Top1: 52.84 | Top2: 76.02 | Top3: 87.21 | Top5: 94.32
Saved best model
Epoch 04 | Train Loss: 1.4355 | Top1: 54.19 | Top2: 76.43 | Top3: 87.03 | Top5: 94.70
Saved best model
Epoch 05 | Train Loss: 1.4066 | Top1: 53.83 | Top2: 75.56 | Top3: 86.91 | Top5: 94.96
Epoch 06 | Train Loss: 1.3851 | Top1: 56.06 | Top2: 77.84 | Top3: 87.82 | Top5: 95.32
Saved best model
Epoch 07 | Train Loss: 1.3616 | Top1: 54.19 | Top2: 76.76 | Top3: 88.09 | Top5: 95.81
Epoch 08 | Train Loss: 1.3692 | Top1: 55.97 | Top2: 77.96 | Top3: 88.26 | Top5: 95.81
Epoch 09 | Train Loss: 1.3495 | Top1: 57.61 | Top2: 79.68 | Top3: 89.64 | Top5: 95.81
Saved best model
Epoch 10 | Train Loss: 1.3347 | Top1: 57.11 | Top2: 79.10 | Top3: 88.38 | Top5: 96.02
Epoch 1

AttributeError: module 'torch.nn' has no attribute 'GeLU'

In [67]:
pos_results_df = pd.DataFrame(pos_results)
pos_results_df = pos_results_df.sort_values(by="Top-1", ascending=False).reset_index(drop=True)
pos_results_df

,Model,Top-1,Top-2,Top-3,Top-5
0,mlp4,58.560140,80.158033,90.869183,97.366111
1,rnn,58.208955,81.913960,90.869183,97.366111
2,ann,57.418788,80.772608,90.693591,96.927129
3,ann_attention,55.838455,80.509219,90.079017,97.102722
4,lstm,53.731343,79.455663,89.201054,96.488147


In [68]:
print("Feature columns:", feature_cols)
print("Label column:", label_col)

print("Feature sample:")
print(train_pos_df[feature_cols].head())

print("Label sample:")
print(train_pos_df[label_col].head())

print("Label unique count:", train_pos_df[label_col].nunique())
print("Label min:", train_pos_df[label_col].min())
print("Label max:", train_pos_df[label_col].max())

Feature columns: ['pos_x', 'pos_y', 'distance', 'sin_angle', 'cos_angle', 'pos_x2', 'pos_y2', 'pos_xy']
Label column: unit1_beam
Feature sample:
      pos_x     pos_y  distance  sin_angle  cos_angle    pos_x2    pos_y2  \
0  0.809288  0.521084  0.962536   0.541365   0.840787  0.654948  0.271528   
1  0.481628  0.294345  0.564450   0.521472   0.853268  0.231965  0.086639   
2  0.220279  0.413660  0.468654   0.882654   0.470023  0.048523  0.171114   
3  0.214123  0.454721  0.502613   0.904714   0.426019  0.045849  0.206772   
4  0.145006  0.409788  0.434688   0.942719   0.333588  0.021027  0.167927   

     pos_xy  
0  0.421707  
1  0.141765  
2  0.091120  
3  0.097366  
4  0.059422  
Label sample:
0    17
1    14
2    17
3    20
4    17
Name: unit1_beam, dtype: int64
Label unique count: 29
Label min: 2
Label max: 30


checking 

In [93]:
"/content/drive/MyDrive/best_position_mlp4.pth"
"/content/drive/MyDrive/best_position_rnn.pth"
"/content/drive/MyDrive/best_position_lstm.pth"

'/content/drive/MyDrive/best_position_lstm.pth'

In [94]:
import os
import torch

ensemble_model_names = ["mlp4", "rnn", "lstm"]

ensemble_models = []

for model_name in ensemble_model_names:
    ckpt_path = f"/content/drive/MyDrive/best_position_{model_name}.pth"
    print(model_name, os.path.exists(ckpt_path), ckpt_path)

    model = build_position_model(
        model_name,
        input_dim=input_dim,
        num_classes=num_classes_pos
    )

    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    model = model.to(device)
    model.eval()

    ensemble_models.append(model)

mlp4 True /content/drive/MyDrive/best_position_mlp4.pth


RuntimeError: Error(s) in loading state_dict for FourLayerMLP:
	size mismatch for net.0.weight: copying a param with shape torch.Size([128, 8]) from checkpoint, the shape in current model is torch.Size([512, 14]).
	size mismatch for net.0.bias: copying a param with shape torch.Size([128]) from checkpoint, the shape in current model is torch.Size([512]).
	size mismatch for net.1.weight: copying a param with shape torch.Size([128]) from checkpoint, the shape in current model is torch.Size([512]).
	size mismatch for net.1.bias: copying a param with shape torch.Size([128]) from checkpoint, the shape in current model is torch.Size([512]).
	size mismatch for net.1.running_mean: copying a param with shape torch.Size([128]) from checkpoint, the shape in current model is torch.Size([512]).
	size mismatch for net.1.running_var: copying a param with shape torch.Size([128]) from checkpoint, the shape in current model is torch.Size([512]).
	size mismatch for net.4.weight: copying a param with shape

In [91]:
print("hello world ")

hello world 


In [92]:
import os
import torch

ensemble_model_names = ["mlp4", "rnn", "lstm"]

ensemble_models = []

for model_name in ensemble_model_names:
    ckpt_path = f"/content/drive/MyDrive/best_position_{model_name}.pth"

    print(model_name, os.path.exists(ckpt_path), ckpt_path)

    model = build_position_model(
        model_name,
        input_dim=input_dim,
        num_classes=num_classes_pos
    )

    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    model = model.to(device)
    model.eval()

    ensemble_models.append(model)

print("Loaded ensemble models:", len(ensemble_models))

mlp4 True /content/drive/MyDrive/best_position_mlp4.pth


RuntimeError: Error(s) in loading state_dict for FourLayerMLP:
	size mismatch for net.0.weight: copying a param with shape torch.Size([128, 8]) from checkpoint, the shape in current model is torch.Size([512, 14]).
	size mismatch for net.0.bias: copying a param with shape torch.Size([128]) from checkpoint, the shape in current model is torch.Size([512]).
	size mismatch for net.1.weight: copying a param with shape torch.Size([128]) from checkpoint, the shape in current model is torch.Size([512]).
	size mismatch for net.1.bias: copying a param with shape torch.Size([128]) from checkpoint, the shape in current model is torch.Size([512]).
	size mismatch for net.1.running_mean: copying a param with shape torch.Size([128]) from checkpoint, the shape in current model is torch.Size([512]).
	size mismatch for net.1.running_var: copying a param with shape torch.Size([128]) from checkpoint, the shape in current model is torch.Size([512]).
	size mismatch for net.4.weight: copying a param with shape